## Deep Research

One of the classic cross-business Agentic use cases! This is huge.

This version uses AnyLLM with OpenRouter's free-model router and a free local web-search function.


<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Commercial implications</h2>
            <span style="color:#00bfff;">A Deep Research agent is broadly applicable to any business area, and to your own day-to-day activities. You can make use of this yourself!
            </span>
        </td>
    </tr>
</table>

In [ ]:
# Imports

from agents import Agent, trace, Runner, function_tool, set_tracing_disabled
from agents.model_settings import ModelSettings
from pydantic import BaseModel
from dotenv import load_dotenv
from ddgs import DDGS
import asyncio
import os
from IPython.display import display, Markdown
from pprint import pprint
import requests

load_dotenv(override=True)

# AnyLLM routes the agents through OpenRouter instead of the OpenAI API.
MODEL = "any-llm/openrouter/openrouter/free"

# OpenAI-hosted tracing requires an OpenAI API key, so disable trace uploads.
set_tracing_disabled(disabled=True)

if not os.getenv("OPENROUTER_API_KEY"):
    raise ValueError(
        "OPENROUTER_API_KEY is missing. Add it to your .env file before running the lab."
    )

# Constants

pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"


## Web search without OpenAI API charges

The original lab used OpenAI's hosted `WebSearchTool`, which requires an OpenAI API call.

This version replaces it with a normal `@function_tool` backed by the free `ddgs` search package. The Agents SDK workflow remains the same, but the search executes locally through the Python package.

### One-time installation

```bash
pip install "openai-agents[any-llm]" python-dotenv ddgs
```

Add an OpenRouter API key to `.env`:

```text
OPENROUTER_API_KEY=your_openrouter_api_key
```

OpenRouter free-model availability and usage limits still apply.


## We will be making 4 Agents:

1. Search Agent, searches online given a search term using a free DDGS function tool
2. Planner Agent, given a query from the user, comes up with searches
3. Report Agent, makes a report from the results
4. Push Agent, sends a notification to the user's phone with a summary

## Our First Agent: Search Agent

Given a search term, search for it on the internet and summarize the results.


In [ ]:
@function_tool
def web_search(query: str) -> str:
    """Search the web and return the five most relevant result titles, URLs, and snippets."""
    try:
        results = DDGS(timeout=15).text(
            query,
            max_results=5,
            safesearch="moderate",
        )
    except Exception as exc:
        return f"Web search failed: {exc}"

    if not results:
        return "No web search results were found."

    formatted_results = []
    for index, item in enumerate(results, start=1):
        title = item.get("title", "Untitled result")
        url = item.get("href", "")
        snippet = item.get("body", "")
        formatted_results.append(
            f"{index}. {title}\nURL: {url}\nSnippet: {snippet}"
        )

    return "\n\n".join(formatted_results)


INSTRUCTIONS = "You are a research assistant. Given a search term, you search the web for that term and \
produce a concise summary of the results. The summary must be 2-3 paragraphs and less than 300 \
words. Capture the main points. Write succinctly, no need to have complete sentences or good \
grammar. This will be consumed by someone synthesizing a report, so it is vital you capture the \
essence and ignore any fluff. Do not include any additional commentary other than the summary itself."

search_agent = Agent(
    name="Search agent",
    instructions=INSTRUCTIONS,
    tools=[web_search],
    model=MODEL,
    model_settings=ModelSettings(tool_choice="required"),
)


In [ ]:
message = "What are the most popular and successful AI Agent frameworks in May 2025"

with trace("Search"):
    result = await Runner.run(search_agent, message)

display(Markdown(result.final_output))

The `trace(...)` context is retained to keep the original lab structure, but OpenAI-hosted trace uploads are disabled.


## Our Second Agent: Planner Agent

Given a query, come up with 5 ideas for web searches that could be run.

Use Structured Outputs as our way to ensure the Agent provides what we need.

In [ ]:
# See note above about cost of WebSearchTool

HOW_MANY_SEARCHES = 5

INSTRUCTIONS = f"You are a helpful research assistant. Given a query, come up with a set of web searches \
to perform to best answer the query. Output {HOW_MANY_SEARCHES} terms to query for."

# We use Pydantic objects to describe the Schema of the output

class WebSearchItem(BaseModel):
    reason: str
    "Your reasoning for why this search is important to the query."

    query: str
    "The search term to use for the web search."


class WebSearchPlan(BaseModel):
    searches: list[WebSearchItem]
    """A list of web searches to perform to best answer the query."""

# We pass in the Pydantic object to ensure the output follows the schema

planner_agent = Agent(
    name="PlannerAgent",
    instructions=INSTRUCTIONS,
    model=MODEL,
    output_type=WebSearchPlan,
)

In [ ]:

message = "What are the most popular and successful AI Agent frameworks in May 2025"

with trace("Search"):
    result = await Runner.run(planner_agent, message)
    pprint(result.final_output)

## Our Third Agent: Writer Agent

Take the results of internet searches and make a report

In [ ]:
INSTRUCTIONS = (
    "You are a senior researcher tasked with writing a cohesive report for a research query. "
    "You will be provided with the original query, and some initial research done by a research assistant.\n"
    "You should first come up with an outline for the report that describes the structure and "
    "flow of the report. Then, generate the report and return that as your final output.\n"
    "The final output should be in markdown format, and it should be lengthy and detailed. Aim "
    "for 5-10 pages of content, at least 1000 words."
)


class ReportData(BaseModel):
    short_summary: str
    """A short 2-3 sentence summary of the findings."""

    markdown_report: str
    """The final report"""

    follow_up_questions: list[str]
    """Suggested topics to research further"""


writer_agent = Agent(
    name="WriterAgent",
    instructions=INSTRUCTIONS,
    model=MODEL,
    output_type=ReportData,
)

## Our Fourth Agent: push notification

Just to show how easy it is to make a tool!

I'm using a nifty product called PushOver - to set this up yourself, visit https://pushover.net

In [ ]:
@function_tool
def push(message: str):
    """Send a push notification with this brief message"""
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)
    return {"status": "success"}

In [ ]:
push

In [ ]:
INSTRUCTIONS = """You are a member of a research team and will be provided with a short summary of a report.
When you receive the report summary, you send a push notification to the user using your tool, informing them that research is complete,
and including the report summary you receive"""


push_agent = Agent(
    name="Push agent",
    instructions=INSTRUCTIONS,
    tools=[push],
    model=MODEL,
    model_settings=ModelSettings(tool_choice="required")
)

### The next 3 functions will plan and execute the search, using planner_agent and search_agent

In [ ]:
async def plan_searches(query: str):
    """ Use the planner_agent to plan which searches to run for the query """
    print("Planning searches...")
    result = await Runner.run(planner_agent, f"Query: {query}")
    print(f"Will perform {len(result.final_output.searches)} searches")
    return result.final_output

async def perform_searches(search_plan: WebSearchPlan):
    """ Call search() for each item in the search plan """
    print("Searching...")
    tasks = [asyncio.create_task(search(item)) for item in search_plan.searches]
    results = await asyncio.gather(*tasks)
    print("Finished searching")
    return results

async def search(item: WebSearchItem):
    """ Use the search agent to run a web search for each item in the search plan """
    input = f"Search term: {item.query}\nReason for searching: {item.reason}"
    result = await Runner.run(search_agent, input)
    return result.final_output

### The next 2 functions write a report and send a push notification

In [ ]:
async def write_report(query: str, search_results: list[str]):
    """ Use the writer agent to write a report based on the search results"""
    print("Thinking about report...")
    input = f"Original query: {query}\nSummarized search results: {search_results}"
    result = await Runner.run(writer_agent, input)
    print("Finished writing report")
    return result.final_output

async def send_push(report: ReportData):
    """ Use the push agent to send a notification to the user """
    print("Pushing...")
    result = await Runner.run(push_agent, report.short_summary)
    print("Push sent")
    return report

### Showtime!

In [ ]:
query ="What are the most popular and successful AI Agent frameworks in May 2025"

with trace("Research trace"):
    print("Starting research...")
    search_plan = await plan_searches(query)
    search_results = await perform_searches(search_plan)
    report = await write_report(query, search_results)
    await send_push(report)  
    print("Hooray!")
display(Markdown(report.markdown_report))

### Trace behavior

OpenAI-hosted tracing is disabled, so no trace is uploaded to the OpenAI Platform.

The workflow still uses the original `trace(...)` blocks, agents, structured outputs, parallel searches, report generation, and optional Pushover notification.
